In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
import os
import torch
import numpy as np
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

monkey_server_path = '/home/carsen/dm11_cluster/fengtongd/Desktop/approxineuro'

In [27]:
# load data
dat = np.load(os.path.join(monkey_server_path, 'data/monkeyv1_cadena_2019.npz'))
images = dat['images']
responses = dat['responses']
real_responses = dat['real_responses']
test_images = dat['test_images']
test_responses = dat['test_responses']
test_real_responses = dat['test_real_responses']
train_idx = dat['train_idx']
val_idx = dat['val_idx']
repetitions = dat['repetitions']
monkey_id = dat['subject_id']
image_ids = dat['image_ids']

# normalize responses
from utils.data import nanarray
responses_nan = nanarray(real_responses, responses)
resp_std = np.nanstd(responses_nan, axis=0) 
responses = responses / resp_std
test_responses = test_responses / resp_std

train_images = images[train_idx]
val_images = images[val_idx]
train_responses = responses[train_idx]
val_responses = responses[val_idx]
train_real_responses = real_responses[train_idx]
val_real_responses = real_responses[val_idx]

print('train:', train_images.shape, train_responses.shape, train_real_responses.shape)
print('val:', val_images.shape, val_responses.shape, val_real_responses.shape)
print('test:', test_images.shape, test_responses.shape, test_real_responses.shape)

print('resp:', responses.min(), responses.max())
print('test resp:', test_responses.min(), test_responses.max())
print('train real resp:', train_real_responses.min(), train_real_responses.max())

train: (18560, 1, 80, 80) (18560, 166) (18560, 166)
val: (4640, 1, 80, 80) (4640, 166) (4640, 166)
test: (1450, 1, 80, 80) (4, 1450, 166) (4, 1450, 166)
resp: 0.0 20.505339
test resp: 0.0 20.87


In [28]:
from utils.data import nanarray
test_responses = nanarray(test_real_responses, test_responses)

monkey_ids = dat['subject_id']
print(len(monkey_ids), np.unique(monkey_ids))

NN = train_responses.shape[1]
Lx, Ly = train_images.shape[2], train_images.shape[3]

166 [ 4 34]


In [29]:
train_images = torch.from_numpy(train_images)
val_images = torch.from_numpy(val_images)
train_responses = torch.from_numpy(train_responses)
val_responses = torch.from_numpy(val_responses)
train_real_responses = torch.from_numpy(train_real_responses)
val_real_responses = torch.from_numpy(val_real_responses)

In [23]:
# build model
from utils import model_builder
seed = 1
nlayers = 1
nconv1 = 192
nconv2 = 192
model, in_channels = model_builder.build_model(NN=166, n_layers=nlayers, n_conv=nconv1, n_conv_mid=nconv2, input_Lx=Lx, input_Ly=Ly)
model_name = model_builder.create_model_name('monkeyV1', '2019', n_layers=nlayers, in_channels=in_channels)
weight_path = os.path.join(monkey_server_path, 'weights', 'fullmodel', 'monkeyV1')
model_path = os.path.join(weight_path, model_name)
print('model path: ', model_path)
model = model.to(device)

input shape of readout:  (192, 40, 40)
model name:  monkeyV1_2019_1layer_192_clamp_sensorium_depthsep_pool.pt
model path:  /home/carsen/dm11_cluster/fengtongd/Desktop/approxineuro/weights/fullmodel/monkeyV1/monkeyV1_2019_1layer_192_clamp_sensorium_depthsep_pool.pt


In [24]:
if not os.path.exists(model_path):
    from utils import model_trainer
    best_state_dict = model_trainer.monkey_train(model, train_responses, train_real_responses, val_responses, val_real_responses, train_images, val_images, device=device)
    torch.save(best_state_dict, model_path)
    print('model saved', model_path)

0.001


KeyboardInterrupt: 

In [ ]:
model.load_state_dict(torch.load(model_path))
print('loaded model', model_path)
model = model.to(device)

loaded model /home/carsen/dm11_cluster/fengtongd/Desktop/approxineuro/weights/fullmodel/monkeyV1/monkeyV1_2019_3layer_192_192_192_clamp_sensorium_depthsep_pool.pt


In [ ]:
from utils import model_trainer
test_images = torch.from_numpy(test_images).to(device)
spks_pred_test = model_trainer.test_epoch(model, test_images)
print('predctions:', spks_pred_test.shape, spks_pred_test.min(), spks_pred_test.max())

predctions: (1450, 166) 0.005456865 5.134309


In [ ]:
from utils import metrics
test_fev, test_feve = metrics.monkey_feve(test_responses, spks_pred_test, repetitions)
print('FEVE (test): ', np.mean(test_feve))

FEVE (test):  0.5603177


/media/carsen/ssd1/approxineuro/notebooks/utils/metrics.py:46: RuntimeWarning: Degrees of freedom <= 0 for slice.
  noise_var.append(np.nanmean(np.nanvar(spks[:repetitions[i], :, i], axis=0, ddof=1)))


In [ ]:
val_images = torch.from_numpy(val_images).to(device)
spks_pred_val = model_trainer.test_epoch(model, val_images)
val_fev, val_feve = metrics.monkey_feve(val_responses, spks_pred_val, repetitions)
print('FEVE (val):', np.mean(val_feve))